# 04 — Teste funcional dos codecs

Executa um round-trip sintético curto em todos os codecs e, opcionalmente, repete o piloto em janelas reais dos corpora.

> Execute as células de cima para baixo. Parâmetros editáveis aparecem em uma célula própria e todo artefato gerado informa seu destino.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    raise RuntimeError("Execute este notebook dentro do repositório CodEco.")

for location in (ROOT / "src", ROOT / "scripts"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))

REPORTS = ROOT / "reports"
DERIVED = ROOT / "data" / "derived"
REPORTS.mkdir(exist_ok=True)
DERIVED.mkdir(exist_ok=True)
print(f"Repositório: {ROOT}")


## Controles

O teste sintético vem ativado: ele confirma que encode, payload e decode realmente funcionam. O piloto nos corpora fica separado porque exige os WAVs instalados. O EnCodec faz parte deste notebook, mas seu primeiro uso baixa o modelo pré-treinado.


In [ ]:
RUN_SMOKE_TEST = True
RUN_CORPUS_PILOT = False
MAX_CLIPS_PER_CORPUS = 1
PILOT_CODECS = ["flac", "mp3", "aac", "opus", "lc3", "encodec"]


In [ ]:
import pandas as pd
import config as C

grid = []
for codec, settings in C.CODECS.items():
    for setting in settings:
        description = ", ".join(f"{key}={value}" for key, value in setting.items())
        grid.append({"codec": codec, "configuração": description,
                     "incluído_no_teste": codec in PILOT_CODECS})
codec_grid = pd.DataFrame(grid)
display(codec_grid)


In [ ]:
import importlib.util
import subprocess
from codecs_lib import make_codec

checks = [("flac", {}, "flac"),
          ("mp3", {"bitrate_kbps": 64}, "libmp3lame"),
          ("aac", {"bitrate_kbps": 64}, "aac"),
          ("opus", {"bitrate_kbps": 24}, "libopus"),
          ("lc3", {"bitrate_kbps": 32}, "liblc3")]
for codec_name, params, encoder_name in checks:
    try:
        codec = make_codec(codec_name, params)
        encoders = subprocess.check_output(
            [str(codec.ffmpeg), "-hide_banner", "-encoders"],
            text=True, stderr=subprocess.STDOUT,
        )
        available = encoder_name in encoders
        print(f"{codec_name:6s} {'disponível' if available else 'ausente':10s} {codec.ffmpeg}")
    except (RuntimeError, FileNotFoundError, subprocess.SubprocessError) as exc:
        print(f"{codec_name:6s} ausente    {exc}")

neural_dependencies = all(importlib.util.find_spec(name) is not None
                          for name in ("torch", "transformers"))
print(f"encodec {'dependências disponíveis' if neural_dependencies else 'bloqueado: instale scripts/requirements.txt'}")


## Round-trip sintético

Esta etapa não produz resultado científico. Ela usa exatamente 4 s em 48 kHz apenas para provar, em cada configuração, que o encoder gera um payload decodificável com o comprimento esperado. Sucessos e falhas aparecem em tabelas separadas; células vazias não são usadas como resultado.


In [ ]:
import gc
import numpy as np

from codecs_lib import align_to_reference, make_codec
from metrics import mel_similarity, snr_db

def synthetic_smoke_signal():
    t = np.arange(C.CLIP_SAMPLES, dtype=np.float32) / C.SR
    tone = 0.28 * np.sin(2 * np.pi * 440 * t) + 0.14 * np.sin(2 * np.pi * 4200 * t)
    chirp = 0.10 * np.sin(2 * np.pi * (900 * t + 1400 * t**2))
    return (tone + chirp).astype(np.float32)

smoke_rows, smoke_failures = [], []
if RUN_SMOKE_TEST:
    x = synthetic_smoke_signal()
    for codec_name in PILOT_CODECS:
        for params in C.CODECS.get(codec_name, []):
            try:
                codec = make_codec(codec_name, params)
                encoded = codec.encode(x)
                decoded = codec.decode(encoded)
                decoded_samples = len(decoded)
                y, lag = align_to_reference(x, decoded)
                smoke_rows.append({
                    "codec": codec_name, "configuração": codec.setting,
                    "payload_bytes": encoded.nbytes,
                    "bitrate_efetivo_kbps": encoded.nbytes * 8 / C.CLIP_SECONDS / 1000,
                    "amostras_decodificadas": decoded_samples, "lag": lag,
                    "snr_db": snr_db(x, y), **mel_similarity(x, y),
                })
                del codec
                gc.collect()
            except Exception as exc:
                smoke_failures.append({"codec": codec_name,
                                       "configuração": ", ".join(f"{k}={v}" for k, v in params.items()),
                                       "erro": f"{type(exc).__name__}: {exc}"})

smoke_results = pd.DataFrame(smoke_rows)
smoke_errors = pd.DataFrame(smoke_failures)
display(smoke_results if not smoke_results.empty else pd.DataFrame({"estado": ["nenhum teste executado"]}))
if not smoke_errors.empty:
    display(smoke_errors)


## Piloto com áudio dos corpora

Ative somente depois de instalar os corpora. Ele repete as mesmas condições em uma janela real por corpus; a avaliação completa e retomável continua no notebook 05.


In [ ]:
if RUN_CORPUS_PILOT:
    from codecs_lib import align_to_reference, make_codec
    from data import load_clip, prepare_all_clip_tables
    from metrics import mel_similarity, snr_db

    # audio_available e path sao propriedades da instalacao atual, nao do
    # freeze. Reconstroi as tabelas para evitar reutilizar caminhos absolutos
    # ou flags de disponibilidade gerados em outra maquina.
    clip_paths = prepare_all_clip_tables()
    availability_rows = []
    for corpus, clip_path in clip_paths.items():
        local_clips = pd.read_csv(clip_path)
        availability_rows.append({"corpus": corpus, "janelas_selecionadas": len(local_clips),
                                  "áudios_disponíveis": int(local_clips.audio_available.astype(bool).sum()),
                                  "tabela": str(clip_path)})
    availability = pd.DataFrame(availability_rows)
    display(availability)
    if availability["áudios_disponíveis"].sum() == 0:
        raise FileNotFoundError(
            "Nenhum WAV do recorte selecionado foi encontrado nesta máquina. "
            "Instale os corpora com ./scripts/install_frozen_corpora.sh all "
            "ou execute este piloto no EASNCloud, dentro do clone que contém data/audio/."
        )

    corpus_rows, corpus_failures = [], []
    for corpus in C.DATASETS:
        clips = pd.read_csv(clip_paths[corpus])
        clips = clips[clips.audio_available.astype(bool)].head(MAX_CLIPS_PER_CORPUS)
        if clips.empty:
            print(f"{corpus}: sem áudio disponível no recorte; corpus ignorado.")
            continue
        for _, clip in clips.iterrows():
            x = load_clip(clip.path, clip.start)
            for codec_name in PILOT_CODECS:
                for params in C.CODECS.get(codec_name, []):
                    setting_label = ", ".join(f"{key}={value}" for key, value in params.items())
                    print(f"Executando {corpus} | {clip.clip_id} | {codec_name} | {setting_label}")
                    try:
                        codec = make_codec(codec_name, params, dataset=corpus)
                        encoded = codec.encode(x)
                        y, lag = align_to_reference(x, codec.decode(encoded))
                        corpus_rows.append({"corpus": corpus, "clip_id": clip.clip_id, "codec": codec_name,
                                            "configuração": codec.setting, "payload_bytes": encoded.nbytes,
                                            "lag": lag, "snr_db": snr_db(x, y), **mel_similarity(x, y)})
                    except Exception as exc:
                        corpus_failures.append({"corpus": corpus, "clip_id": clip.clip_id,
                                                "codec": codec_name, "configuração": str(params),
                                                "erro": f"{type(exc).__name__}: {exc}"})
    corpus_pilot = pd.DataFrame(corpus_rows)
    corpus_errors = pd.DataFrame(corpus_failures)
    corpus_pilot.to_csv(DERIVED / "codec_pilot.csv", index=False)
    display(corpus_pilot)
    if not corpus_errors.empty:
        display(corpus_errors)
else:
    print("Piloto dos corpora não executado. Defina RUN_CORPUS_PILOT=True após instalar os WAVs.")
